# VQA Tiếng Việt - Hướng A: ResNet50 + PhoBERT + LSTM/Transformer Decoder

**Kiến trúc:**
- Image Encoder: ResNet50 (pretrained, frozen backbone)
- Text Encoder: PhoBERT (vinai/phobert-base)
- Fusion: Concat + Fully Connected
- **A1**: LSTM Decoder
- **A2**: Transformer Decoder

**Metrics:** VQA Accuracy, BLEU, ROUGE-L, METEOR, BERTScore

## 1. Cài đặt thư viện

In [21]:
!pip install -q transformers sentencepiece evaluate rouge_score bert_score nltk underthesea accelerate
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('Cài đặt hoàn tất!')


Cài đặt hoàn tất!


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Import & Cấu hình

In [23]:
import os, json, re, glob, math, copy, time
import numpy as np
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision.transforms as T
import torchvision.models as models

import warnings
import logging
warnings.filterwarnings('ignore')           # tắt toàn bộ Python warnings

from transformers import AutoTokenizer, AutoModel
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()            # tắt verbose của HuggingFace
import evaluate
# Tắt evaluate progress bar
evaluate.logging.set_verbosity_error()
from bert_score import score as bert_score_fn
from tqdm.auto import tqdm

# --- Paths ------------------------------------------------------------------
IMAGE_ROOT   = "/content/drive/MyDrive/task1/images"       # train/ val/ test/
ANNOT_ROOT   = "/content/drive/MyDrive/task1/annotations"  # train.json val.json test.json
CKPT_ROOT    = "/content/drive/MyDrive/task1/checkpoints"  # A1/ A2/
RESULT_ROOT  = "/content/drive/MyDrive/task1/results"

os.makedirs(f"{CKPT_ROOT}/A1",        exist_ok=True)
os.makedirs(f"{CKPT_ROOT}/A2",        exist_ok=True)
os.makedirs(f"{RESULT_ROOT}/metrics", exist_ok=True)
os.makedirs(f"{RESULT_ROOT}/plots",   exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# --- Hyper-parameters --------------------------------------------------------
class Config:
    # Chung
    PHOBERT_NAME    = "vinai/phobert-base"  # PhoBERT base
    IMAGE_SIZE      = 224
    IMG_FEAT_DIM    = 2048   # ResNet50 avgpool output
    TXT_FEAT_DIM    = 768    # PhoBERT CLS hidden size
    FUSION_DIM      = 1024   # After concat + Linear
    EMBED_DIM       = 256    # Decoder token embedding dim
    MAX_ANS_LEN     = 20     # Max answer tokens (decode)
    MAX_Q_LEN       = 64     # Max question tokens (PhoBERT)

    # LSTM decoder (A1)
    LSTM_HIDDEN     = 512
    LSTM_LAYERS     = 2
    LSTM_DROPOUT    = 0.3

    # Transformer decoder (A2)
    TF_NHEAD        = 8
    TF_NUM_LAYERS   = 3
    TF_FF_DIM       = 2048
    TF_DROPOUT      = 0.1

    # Training
    BATCH_SIZE      = 32
    EPOCHS          = 5
    LR              = 3e-4
    WEIGHT_DECAY    = 1e-4
    PHOBERT_LR      = 2e-5   # riêng cho PhoBERT (fine-tune nhẹ)
    SAVE_EVERY      = 3    # luu checkpoint moi 3 epoch (tiet kiem dia)
    CLIP_GRAD       = 1.0
    TEACHER_FORCING = 0.5    # xác suất teacher-forcing trong training

cfg = Config()
print("Cấu hình xong!")

Device: cuda
Cấu hình xong!


## 3. Xây dựng Vocabulary từ tập Train

In [24]:
# --- Vocabulary -------------------------------------------------------------
class Vocabulary:
    """Vocabulary cho phần decoder (answer generation).
    Tokenize ở mức từ (word-level), đủ cho câu trả lời ngắn."""

    PAD = "<PAD>"
    SOS = "<SOS>"
    EOS = "<EOS>"
    UNK = "<UNK>"
    SPECIALS = [PAD, SOS, EOS, UNK]

    def __init__(self):
        self.word2idx = {}
        self.idx2word = {}
        self._build_specials()

    def _build_specials(self):
        for i, w in enumerate(self.SPECIALS):
            self.word2idx[w] = i
            self.idx2word[i] = w

    def build_from_answers(self, answers: list, min_freq: int = 1):
        """Xây vocab từ danh sách câu trả lời."""
        counter = Counter()
        for ans in answers:
            for token in self._tokenize(ans):
                counter[token] += 1
        for word, freq in counter.items():
            if freq >= min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
        print(f"Vocabulary size: {len(self.word2idx)} tokens")

    @staticmethod
    def _tokenize(text: str) -> list:
        """Tokenize đơn giản theo khoảng trắng + lowercase."""
        return text.lower().strip().split()

    def encode(self, text: str, max_len: int = None) -> list:
        tokens = self._tokenize(text)
        ids = [self.word2idx.get(t, self.word2idx[self.UNK]) for t in tokens]
        ids = [self.word2idx[self.SOS]] + ids + [self.word2idx[self.EOS]]
        if max_len:
            ids = ids[:max_len]
        return ids

    def decode(self, ids: list, skip_special: bool = True) -> str:
        words = []
        for i in ids:
            w = self.idx2word.get(i, self.UNK)
            if skip_special and w in self.SPECIALS:
                if w == self.EOS:
                    break
                continue
            words.append(w)
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)

    @property
    def pad_idx(self): return self.word2idx[self.PAD]
    @property
    def sos_idx(self): return self.word2idx[self.SOS]
    @property
    def eos_idx(self): return self.word2idx[self.EOS]


# --- Xây dựng vocab từ train.json -------------------------------------------
train_json_path = os.path.join(ANNOT_ROOT, "train.json")
with open(train_json_path, encoding="utf-8") as f:
    train_data_raw = json.load(f)

all_answers = [item["answer"] for item in train_data_raw]
vocab = Vocabulary()
vocab.build_from_answers(all_answers, min_freq=1)
print(f"Tổng số token: {len(vocab)}")

Vocabulary size: 586 tokens
Tổng số token: 586


## 4. Dataset & DataLoader

In [25]:
# --- Image transforms -------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = T.Compose([
    T.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


# --- VQA Dataset ------------------------------------------------------------
class VQADatasetA(Dataset):
    """Dataset cho Hướng A - trả về image tensor, question tokens, answer tokens."""

    def __init__(
        self,
        json_path: str,
        image_dir: str,
        vocab: Vocabulary,
        tokenizer,           # PhoBERT tokenizer
        transform=None,
        max_q_len: int = 64,
        max_a_len: int = 22, # bao gồm SOS + EOS
    ):
        with open(json_path, encoding="utf-8") as f:
            self.data = json.load(f)
        self.image_dir = image_dir
        self.vocab     = vocab
        self.tokenizer = tokenizer
        self.transform = transform or val_transform
        self.max_q_len = max_q_len
        self.max_a_len = max_a_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # --- Image ---
        img_path = os.path.join(self.image_dir, item["image_id"])
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (cfg.IMAGE_SIZE, cfg.IMAGE_SIZE))
        image = self.transform(image)  # (3, H, W)

        # --- Question -> PhoBERT tokens ---
        q_enc = self.tokenizer(
            item["question"],
            max_length=self.max_q_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        q_ids      = q_enc["input_ids"].squeeze(0)       # (max_q_len,)
        q_attn     = q_enc["attention_mask"].squeeze(0)  # (max_q_len,)

        # --- Answer -> Vocab ids ---
        ans_ids = self.vocab.encode(item["answer"], max_len=self.max_a_len)
        # Pad đến max_a_len
        ans_ids += [self.vocab.pad_idx] * (self.max_a_len - len(ans_ids))
        ans_tensor = torch.tensor(ans_ids, dtype=torch.long)  # (max_a_len,)

        # Lưu raw answer cho evaluation
        raw_answer = item["answer"]

        return {
            "image":      image,
            "q_ids":      q_ids,
            "q_attn":     q_attn,
            "ans_ids":    ans_tensor,
            "raw_answer": raw_answer,
            "raw_question": item["question"],
        }


# --- Load PhoBERT Tokenizer -------------------------------------------------
print("Đang load PhoBERT tokenizer...")
phobert_tokenizer = AutoTokenizer.from_pretrained(cfg.PHOBERT_NAME)

# --- Build DataLoaders ------------------------------------------------------
def make_loaders(cfg, vocab, tokenizer):
    train_ds = VQADatasetA(
        json_path  = os.path.join(ANNOT_ROOT, "train.json"),
        image_dir  = os.path.join(IMAGE_ROOT, "train"),
        vocab      = vocab,
        tokenizer  = tokenizer,
        transform  = train_transform,
        max_q_len  = cfg.MAX_Q_LEN,
        max_a_len  = cfg.MAX_ANS_LEN + 2,
    )
    val_ds = VQADatasetA(
        json_path  = os.path.join(ANNOT_ROOT, "val.json"),
        image_dir  = os.path.join(IMAGE_ROOT, "val"),
        vocab      = vocab,
        tokenizer  = tokenizer,
        transform  = val_transform,
        max_q_len  = cfg.MAX_Q_LEN,
        max_a_len  = cfg.MAX_ANS_LEN + 2,
    )
    test_ds = VQADatasetA(
        json_path  = os.path.join(ANNOT_ROOT, "test.json"),
        image_dir  = os.path.join(IMAGE_ROOT, "test"),
        vocab      = vocab,
        tokenizer  = tokenizer,
        transform  = val_transform,
        max_q_len  = cfg.MAX_Q_LEN,
        max_a_len  = cfg.MAX_ANS_LEN + 2,
    )
    train_loader = DataLoader(
        train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
        num_workers=0, pin_memory=False
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=False
    )
    test_loader = DataLoader(
        test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=False
    )
    print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders(cfg, vocab, phobert_tokenizer)
print("DataLoaders sẵn sàng!")

Đang load PhoBERT tokenizer...
Train: 13302 | Val: 3715 | Test: 1857
DataLoaders sẵn sàng!


## 5. Kiến trúc mô hình - Shared Encoder + Fusion

In [26]:
# --- Image Encoder: ResNet50 -------------------------------------------------
class ImageEncoder(nn.Module):
    """ResNet50 pretrained, bỏ lớp FC cuối, lấy output avgpool (2048-dim)."""

    def __init__(self, fine_tune_last_n_blocks: int = 1):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        # Bỏ avgpool + fc -> lấy thủ công
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])  # -> (B, 2048, 7, 7)
        self.pool     = nn.AdaptiveAvgPool2d((1, 1))                  # -> (B, 2048, 1, 1)

        # Freeze toàn bộ, chỉ unfreeze `fine_tune_last_n_blocks` layer4-blocks
        for param in self.backbone.parameters():
            param.requires_grad = False

        if fine_tune_last_n_blocks > 0:
            # layer4 của ResNet50 có 3 blocks
            layer4 = list(self.backbone[-1].children())  # layer4's blocks
            for block in layer4[-fine_tune_last_n_blocks:]:
                for p in block.parameters():
                    p.requires_grad = True

    def forward(self, x):  # x: (B, 3, 224, 224)
        feat = self.backbone(x)   # (B, 2048, 7, 7)
        feat = self.pool(feat)    # (B, 2048, 1, 1)
        feat = feat.flatten(1)    # (B, 2048)
        return feat


# --- Text Encoder: PhoBERT ----------------------------------------------------
class TextEncoder(nn.Module):
    """PhoBERT, lấy [CLS] hidden state làm đại diện câu hỏi (768-dim)."""

    def __init__(self, model_name: str, fine_tune: bool = True):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        if not fine_tune:
            for p in self.bert.parameters():
                p.requires_grad = False

    def forward(self, input_ids, attention_mask):  # -> (B, 768)
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # [CLS] token
        return cls


# --- Fusion Module ------------------------------------------------------------
class FusionModule(nn.Module):
    """Concat(img_feat, txt_feat) -> Linear -> ReLU -> Dropout -> fusion_feat."""

    def __init__(self, img_dim: int, txt_dim: int, fusion_dim: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim + txt_dim, fusion_dim * 2),
            nn.LayerNorm(fusion_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
        )

    def forward(self, img_feat, txt_feat):  # -> (B, fusion_dim)
        combined = torch.cat([img_feat, txt_feat], dim=-1)
        return self.net(combined)

print("Đã định nghĩa ImageEncoder, TextEncoder, FusionModule")

Đã định nghĩa ImageEncoder, TextEncoder, FusionModule


## 6. A1 - LSTM Decoder

In [27]:
# --- A1: LSTM Decoder --------------------------------------------------------
class LSTMDecoder(nn.Module):
    """
    Seq2Seq LSTM Decoder cho câu trả lời.
    - Context vector từ Fusion được dùng để khởi tạo hidden & cell state.
    - Ở mỗi bước: input = embedding(prev_token) + context (Attention-free, đơn giản).
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        fusion_dim: int,
        num_layers: int = 2,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.input_proj = nn.Linear(embed_dim + fusion_dim, embed_dim)  # concat + proj
        self.lstm       = nn.LSTM(
            input_size  = embed_dim,
            hidden_size = hidden_dim,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.h_proj     = nn.Linear(fusion_dim, hidden_dim * num_layers)
        self.c_proj     = nn.Linear(fusion_dim, hidden_dim * num_layers)
        self.dropout    = nn.Dropout(dropout)
        self.fc_out     = nn.Linear(hidden_dim, vocab_size)

    def _init_hidden(self, context):  # context: (B, fusion_dim)
        B = context.size(0)
        h = self.h_proj(context)  # (B, hidden_dim * num_layers)
        c = self.c_proj(context)
        # Reshape -> (num_layers, B, hidden_dim)
        h = h.view(B, self.num_layers, self.hidden_dim).permute(1, 0, 2).contiguous()
        c = c.view(B, self.num_layers, self.hidden_dim).permute(1, 0, 2).contiguous()
        return h, c

    def forward(self, context, target_ids=None, teacher_forcing_ratio: float = 0.5):
        """
        Training (target_ids != None): Teacher-forcing.
        Inference (target_ids is None): Greedy decode.
        context:    (B, fusion_dim)
        target_ids: (B, max_len) gồm SOS ... EOS
        Returns: logits (B, max_len-1, vocab_size)
        """
        B = context.size(0)
        h, c = self._init_hidden(context)

        if target_ids is not None:
            # Training: teacher-forcing
            max_len = target_ids.size(1) - 1  # bỏ EOS ở cuối
            all_logits = []
            inp = target_ids[:, 0]  # SOS token

            for t in range(max_len):
                emb = self.embedding(inp)           # (B, embed_dim)
                inp_combined = torch.cat([emb, context], dim=-1)  # (B, embed+fusion)
                inp_proj = self.input_proj(inp_combined).unsqueeze(1)  # (B,1,embed)
                out, (h, c) = self.lstm(inp_proj, (h, c))
                logit = self.fc_out(self.dropout(out.squeeze(1)))  # (B, vocab)
                all_logits.append(logit)

                # Teacher forcing
                use_teacher = torch.rand(1).item() < teacher_forcing_ratio
                if use_teacher:
                    inp = target_ids[:, t + 1]
                else:
                    inp = logit.argmax(dim=-1)

            return torch.stack(all_logits, dim=1)  # (B, max_len, vocab)

        else:
            # Inference: greedy decode
            sos = torch.tensor([vocab.sos_idx] * B, device=context.device)
            inp = sos
            results = []
            done = torch.zeros(B, dtype=torch.bool, device=context.device)

            for _ in range(cfg.MAX_ANS_LEN):
                emb = self.embedding(inp)
                inp_combined = torch.cat([emb, context], dim=-1)
                inp_proj = self.input_proj(inp_combined).unsqueeze(1)
                out, (h, c) = self.lstm(inp_proj, (h, c))
                logit = self.fc_out(self.dropout(out.squeeze(1)))
                pred = logit.argmax(dim=-1)  # (B,)
                results.append(pred)
                done = done | (pred == vocab.eos_idx)
                inp = pred
                if done.all():
                    break

            return torch.stack(results, dim=1)  # (B, gen_len)


# --- A1 Full Model ------------------------------------------------------------
class VQA_A1(nn.Module):
    """Hướng A1: ResNet50 + PhoBERT + Concat+FC Fusion + LSTM Decoder."""

    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.image_enc = ImageEncoder(fine_tune_last_n_blocks=1)
        self.text_enc  = TextEncoder(cfg.PHOBERT_NAME, fine_tune=True)
        self.fusion    = FusionModule(cfg.IMG_FEAT_DIM, cfg.TXT_FEAT_DIM, cfg.FUSION_DIM)
        self.decoder   = LSTMDecoder(
            vocab_size  = vocab_size,
            embed_dim   = cfg.EMBED_DIM,
            hidden_dim  = cfg.LSTM_HIDDEN,
            fusion_dim  = cfg.FUSION_DIM,
            num_layers  = cfg.LSTM_LAYERS,
            dropout     = cfg.LSTM_DROPOUT,
        )

    def forward(self, image, q_ids, q_attn, target_ids=None, teacher_forcing_ratio=0.5):
        img_feat = self.image_enc(image)              # (B, 2048)
        txt_feat = self.text_enc(q_ids, q_attn)       # (B, 768)
        ctx      = self.fusion(img_feat, txt_feat)    # (B, 1024)
        return self.decoder(ctx, target_ids, teacher_forcing_ratio)

print("Đã định nghĩa VQA_A1 (LSTM Decoder)")

Đã định nghĩa VQA_A1 (LSTM Decoder)


## 7. A2 - Transformer Decoder

In [28]:
# --- Positional Encoding -----------------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 100, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):  # x: (B, T, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# --- A2: Transformer Decoder --------------------------------------------------
class TransformerDecoder(nn.Module):
    """
    Transformer Decoder cho câu trả lời.
    - Memory = context (B, 1, fusion_dim) từ Fusion module.
    - Causal mask để tránh nhìn tương lai.
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        fusion_dim: int,
        nhead: int = 8,
        num_layers: int = 3,
        ff_dim: int = 2048,
        dropout: float = 0.1,
        max_len: int = 50,
    ):
        super().__init__()
        self.embed_dim = embed_dim

        self.embedding    = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_enc      = PositionalEncoding(embed_dim, max_len, dropout)
        self.mem_proj     = nn.Linear(fusion_dim, embed_dim)  # project memory -> embed_dim

        decoder_layer = nn.TransformerDecoderLayer(
            d_model         = embed_dim,
            nhead           = nhead,
            dim_feedforward = ff_dim,
            dropout         = dropout,
            batch_first     = True,  # (B, T, D)
            norm_first      = True,  # Pre-LN cho stable training
        )
        self.transformer_dec = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out          = nn.Linear(embed_dim, vocab_size)
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.embedding.weight, 0, 0.02)
        nn.init.xavier_uniform_(self.fc_out.weight)

    def _causal_mask(self, size, device):
        """Upper-triangular mask (float -inf) để không nhìn token tương lai."""
        mask = torch.triu(torch.ones(size, size, device=device), diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))

    def forward(self, context, target_ids=None, teacher_forcing_ratio: float = 0.5):
        """
        Training: trả về logits (B, max_len-1, vocab_size).
        Inference: trả về token ids (B, gen_len).
        """
        B = context.size(0)
        memory = self.mem_proj(context).unsqueeze(1)  # (B, 1, embed_dim)

        if target_ids is not None:
            # Teacher-forcing (full sequence decode)
            tgt_in = target_ids[:, :-1]              # (B, T-1) - bỏ EOS
            T = tgt_in.size(1)
            tgt_emb = self.pos_enc(self.embedding(tgt_in) * math.sqrt(self.embed_dim))
            causal  = self._causal_mask(T, tgt_in.device)
            pad_mask = (tgt_in == 0)                  # True ở vị trí PAD
            out = self.transformer_dec(
                tgt          = tgt_emb,
                memory       = memory,
                tgt_mask     = causal,
                tgt_key_padding_mask = pad_mask,
            )  # (B, T-1, embed_dim)
            logits = self.fc_out(out)  # (B, T-1, vocab_size)
            return logits

        else:
            # Autoregressive greedy decode
            generated = torch.full((B, 1), vocab.sos_idx, dtype=torch.long, device=context.device)
            done = torch.zeros(B, dtype=torch.bool, device=context.device)

            for _ in range(cfg.MAX_ANS_LEN):
                T = generated.size(1)
                tgt_emb = self.pos_enc(self.embedding(generated) * math.sqrt(self.embed_dim))
                causal  = self._causal_mask(T, generated.device)
                out = self.transformer_dec(
                    tgt    = tgt_emb,
                    memory = memory,
                    tgt_mask = causal,
                )
                logit     = self.fc_out(out[:, -1, :])  # (B, vocab)
                next_tok  = logit.argmax(dim=-1, keepdim=True)  # (B, 1)
                generated = torch.cat([generated, next_tok], dim=1)
                done = done | (next_tok.squeeze(-1) == vocab.eos_idx)
                if done.all():
                    break

            return generated[:, 1:]  # bỏ SOS token đầu


# --- A2 Full Model ------------------------------------------------------------
class VQA_A2(nn.Module):
    """Hướng A2: ResNet50 + PhoBERT + Concat+FC Fusion + Transformer Decoder."""

    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.image_enc = ImageEncoder(fine_tune_last_n_blocks=1)
        self.text_enc  = TextEncoder(cfg.PHOBERT_NAME, fine_tune=True)
        self.fusion    = FusionModule(cfg.IMG_FEAT_DIM, cfg.TXT_FEAT_DIM, cfg.FUSION_DIM)
        self.decoder   = TransformerDecoder(
            vocab_size = vocab_size,
            embed_dim  = cfg.EMBED_DIM,
            fusion_dim = cfg.FUSION_DIM,
            nhead      = cfg.TF_NHEAD,
            num_layers = cfg.TF_NUM_LAYERS,
            ff_dim     = cfg.TF_FF_DIM,
            dropout    = cfg.TF_DROPOUT,
        )

    def forward(self, image, q_ids, q_attn, target_ids=None, teacher_forcing_ratio=0.5):
        img_feat = self.image_enc(image)
        txt_feat = self.text_enc(q_ids, q_attn)
        ctx      = self.fusion(img_feat, txt_feat)
        return self.decoder(ctx, target_ids, teacher_forcing_ratio)

print("Đã định nghĩa VQA_A2 (Transformer Decoder)")

Đã định nghĩa VQA_A2 (Transformer Decoder)


## 8. Checkpoint Manager (Resume Training)

In [29]:
import shutil

class CheckpointManager:
    """
    Quan ly luu/load checkpoint, ho tro resume training.
    Chi giu lai toi da MAX_KEEP checkpoint gan nhat de tiet kiem dung luong.
    """

    MAX_KEEP = 2          # so luong ckpt epoch giu lai toi da
    MIN_FREE_GB = 1.0     # nguong dung luong trong toi thieu (GB) truoc khi luu

    @staticmethod
    def _free_gb(path: str) -> float:
        """Kiem tra dung luong dia con trong (GB) tai thu muc chua path."""
        usage = shutil.disk_usage(os.path.dirname(os.path.abspath(path)))
        return usage.free / (1024 ** 3)

    @staticmethod
    def _cleanup_old_ckpts(ckpt_dir: str):
        """Xoa cac checkpoint epoch cu, chi giu MAX_KEEP file moi nhat."""
        ckpts = sorted(
            glob.glob(os.path.join(ckpt_dir, "ckpt_epoch_*.pth")),
            key=lambda x: int(re.search(r"epoch_(\d+)", x).group(1))
        )
        # Giu lai MAX_KEEP cuoi, xoa phan con lai
        to_delete = ckpts[:-CheckpointManager.MAX_KEEP] if len(ckpts) > CheckpointManager.MAX_KEEP else []
        for f in to_delete:
            try:
                os.remove(f)
                print(f"  [XOA] {os.path.basename(f)}")
            except OSError:
                pass

    @staticmethod
    def save(model, optimizer, scheduler, epoch, val_loss, best_loss,
             ckpt_dir: str, log: dict, is_best: bool = False):
        os.makedirs(ckpt_dir, exist_ok=True)

        # Kiem tra dung luong truoc khi luu
        free_gb = CheckpointManager._free_gb(ckpt_dir)
        if free_gb < CheckpointManager.MIN_FREE_GB:
            print(f"  [CANH BAO] Dia chi con {free_gb:.2f} GB - bo qua luu checkpoint epoch.")
            # Van luu log de theo doi
            log_path = os.path.join(ckpt_dir, "training_log.json")
            with open(log_path, "a", encoding="utf-8") as flog:
                flog.write(json.dumps(log, ensure_ascii=False) + "\n")
            return

        # Checkpoint epoch: luu toan bo trang thai de co the resume
        state = {
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
            "val_loss":             val_loss,
            "best_loss":            best_loss,
        }
        path = os.path.join(ckpt_dir, f"ckpt_epoch_{epoch:03d}.pth")
        try:
            torch.save(state, path)
            print(f"  [LUU] {os.path.basename(path)} | Dia con: {free_gb:.1f} GB")
        except RuntimeError as e:
            print(f"  [LOI LUU] {e} - Bo qua checkpoint epoch nay.")
            # Xoa file bi loi neu ton tai
            if os.path.exists(path):
                os.remove(path)

        # Xoa ckpt cu sau khi luu thanh cong
        CheckpointManager._cleanup_old_ckpts(ckpt_dir)

        # Best model: chi luu weights (khong co optimizer) -> nhe hon
        if is_best:
            best_path = os.path.join(ckpt_dir, "best_model.pth")
            best_state = {
                "epoch":                epoch,
                "model_state_dict":     model.state_dict(),
                "val_loss":             val_loss,
                "best_loss":            best_loss,
            }
            try:
                torch.save(best_state, best_path)
                print(f"  [TOT NHAT] Mo hinh tot nhat! Val Loss: {val_loss:.4f}")
            except RuntimeError as e:
                print(f"  [LOI LUU BEST] {e}")

        # Luu log
        log_path = os.path.join(ckpt_dir, "training_log.json")
        with open(log_path, "a", encoding="utf-8") as flog:
            flog.write(json.dumps(log, ensure_ascii=False) + "\n")

    @staticmethod
    def load_latest(model, optimizer, scheduler, ckpt_dir: str, device):
        if not os.path.exists(ckpt_dir):
            print(f"  {ckpt_dir} chua ton tai -> Train tu dau.")
            return 0, float("inf")

        ckpts = glob.glob(os.path.join(ckpt_dir, "ckpt_epoch_*.pth"))
        if not ckpts:
            print(f"  Khong tim thay checkpoint -> Train tu dau.")
            return 0, float("inf")

        ckpts.sort(key=lambda x: int(re.search(r"epoch_(\d+)", x).group(1)))
        latest = ckpts[-1]

        try:
            state = torch.load(latest, map_location=device, weights_only=False)
        except Exception as e:
            print(f"  [LOI DOC CKPT] {e} -> Train tu dau.")
            return 0, float("inf")

        model.load_state_dict(state["model_state_dict"])
        if optimizer and state.get("optimizer_state_dict"):
            optimizer.load_state_dict(state["optimizer_state_dict"])
        if scheduler and state.get("scheduler_state_dict"):
            scheduler.load_state_dict(state["scheduler_state_dict"])

        start_epoch = state["epoch"] + 1
        best_loss   = state.get("best_loss", float("inf"))
        print(f"  [RESUME] Tiep tuc tu epoch {start_epoch} | Best Loss: {best_loss:.4f}")
        return start_epoch, best_loss

print("CheckpointManager san sang!")


CheckpointManager san sang!


## 9. Metrics Evaluation

In [30]:
from datetime import datetime
import os
import json
import warnings
import evaluate

class UniversalEvaluator:
    def __init__(self, base_metrics_dir):
        # Lưu đường dẫn gốc thay vì tạo thư mục ngay lập tức
        self.base_metrics_dir = base_metrics_dir

        print("Đang tải các công cụ chấm điểm (Metrics)...")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            self.bleu_metric = evaluate.load("bleu")
            self.rouge_metric = evaluate.load("rouge")
            self.meteor_metric = evaluate.load("meteor")
            self.bertscore_metric = evaluate.load("bertscore")
        print("Đã tải xong bộ chấm điểm!")

    def evaluate_and_save(self, all_preds, all_labels, model_name):
        """
        Tính toán metrics và tự động lưu lịch sử JSON cho mô hình vào thư mục riêng (A1 hoặc A2).
        """
        print(f"\nĐANG CHẤM ĐIỂM CHO MÔ HÌNH: {model_name}...")

        valid_pairs = [(p, l) for p, l in zip(all_preds, all_labels) if p.strip()]
        if not valid_pairs:
            print("Cảnh báo: Không có prediction nào hợp lệ!")
            return {"metrics": {"vqa_accuracy": 0, "bleu": 0, "rougeL": 0, "meteor": 0, "bertscore_f1": 0}}

        valid_preds, valid_labels = zip(*valid_pairs)
        valid_preds, valid_labels = list(valid_preds), list(valid_labels)

        # 1. Exact Match (VQA Accuracy - Trùng khớp 100%)
        exact_matches = sum([1 if p.lower().strip() == l.lower().strip() else 0 for p, l in zip(all_preds, all_labels)])
        vqa_acc = exact_matches / len(all_preds) if len(all_preds) > 0 else 0

        # 2. BLEU, ROUGE, METEOR
        try:
            bleu_score = self.bleu_metric.compute(predictions=valid_preds, references=[[l] for l in valid_labels])
            bleu_val = bleu_score['bleu']
        except Exception:
            bleu_val = 0.0

        rouge_score = self.rouge_metric.compute(predictions=valid_preds, references=valid_labels)
        meteor_score = self.meteor_metric.compute(predictions=valid_preds, references=valid_labels)

        # 3. BERTScore (Ngữ nghĩa tiếng Việt)
        print("   -> Đang chạy BERTScore (có thể mất vài chục giây)...")
        bert_res = self.bertscore_metric.compute(predictions=valid_preds, references=valid_labels, lang="vi")
        bert_f1 = sum(bert_res['f1']) / len(bert_res['f1'])

        results = {
            "model": model_name,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "metrics": {
                "vqa_accuracy": round(vqa_acc * 100, 2),
                "bleu": round(bleu_val * 100, 2),
                "rougeL": round(rouge_score['rougeL'] * 100, 2),
                "meteor": round(meteor_score['meteor'] * 100, 2),
                "bertscore_f1": round(bert_f1 * 100, 2)
            }
        }

        print(f"\nBÁO CÁO THÀNH TÍCH ({model_name}):")
        for k, v in results["metrics"].items():
            print(f"  -> {k.upper()}: {v}")

        # --- TẠO THƯ MỤC LƯU THEO TÊN MÔ HÌNH (A1 / A2) ---
        model_metrics_dir = os.path.join(self.base_metrics_dir, model_name)
        os.makedirs(model_metrics_dir, exist_ok=True)

        save_path = os.path.join(model_metrics_dir, f"evaluation_results_{model_name}.json")

        history = []
        if os.path.exists(save_path):
            with open(save_path, 'r', encoding='utf-8') as f:
                history = json.load(f)

        history.append(results)

        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(history, f, ensure_ascii=False, indent=4)

        print(f"Đã lưu kết quả an toàn tại: {save_path}")
        return results

def decode_predictions(pred_ids_batch, vocab: Vocabulary):
    """Chuyển tensor ids -> list of strings."""
    decoded = []
    for ids in pred_ids_batch:
        decoded.append(vocab.decode(ids.tolist()))
    return decoded

print("Metrics functions & UniversalEvaluator sẵn sàng!")

Metrics functions & UniversalEvaluator sẵn sàng!


## 10. Training Loop (chung cho A1 & A2)

In [31]:
import evaluate
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import warnings

# Load sẵn các metrics (chỉ load 1 lần để tránh tốn thời gian mỗi epoch)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _bleu = evaluate.load("bleu")
    _rouge = evaluate.load("rouge")
    _meteor = evaluate.load("meteor")
    _bertscore = evaluate.load("bertscore")

def compute_all_metrics(all_preds, all_refs, use_bertscore=False):
    """Hàm tính toán nhanh các metrics trong quá trình training (validation)"""
    valid_pairs = [(p, l) for p, l in zip(all_preds, all_refs) if p.strip()]
    if not valid_pairs:
        return {"vqa_accuracy": 0.0, "bleu": 0.0, "rougeL": 0.0, "meteor": 0.0, "bertscore_f1": 0.0}

    valid_preds, valid_labels = zip(*valid_pairs)
    valid_preds, valid_labels = list(valid_preds), list(valid_labels)

    # 1. Exact Match (VQA Accuracy)
    exact_matches = sum([1 if p.lower().strip() == l.lower().strip() else 0 for p, l in zip(all_preds, all_refs)])
    vqa_acc = exact_matches / len(all_preds) if len(all_preds) > 0 else 0.0

    # 2. BLEU, ROUGE, METEOR
    try:
        bleu_val = _bleu.compute(predictions=valid_preds, references=[[l] for l in valid_labels])['bleu']
    except Exception:
        bleu_val = 0.0

    rouge_val = _rouge.compute(predictions=valid_preds, references=valid_labels)['rougeL']
    meteor_val = _meteor.compute(predictions=valid_preds, references=valid_labels)['meteor']

    # 3. BERTScore (Option: tắt lúc train để đỡ chậm, bật ở test)
    bert_f1 = 0.0
    if use_bertscore:
        bert_res = _bertscore.compute(predictions=valid_preds, references=valid_labels, lang="vi")
        bert_f1 = sum(bert_res['f1']) / len(bert_res['f1'])

    return {
        "vqa_accuracy": vqa_acc,
        "bleu": bleu_val,
        "rougeL": rouge_val,
        "meteor": meteor_val,
        "bertscore_f1": bert_f1
    }

def train_one_epoch(model, loader, optimizer, criterion, cfg, device, tf_ratio):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc="  Train", leave=False)

    for batch in pbar:
        image   = batch["image"].to(device)
        q_ids   = batch["q_ids"].to(device)
        q_attn  = batch["q_attn"].to(device)
        ans_ids = batch["ans_ids"].to(device)  # (B, max_a_len)

        optimizer.zero_grad()

        # Forward - logits: (B, max_len-1, vocab_size)
        logits = model(image, q_ids, q_attn, target_ids=ans_ids,
                       teacher_forcing_ratio=tf_ratio)

        # Target: ans_ids[:, 1:] -> bỏ SOS đầu, giữ EOS cuối
        target = ans_ids[:, 1:].reshape(-1)  # (B*(max_len-1),)
        logits_flat = logits.reshape(-1, logits.size(-1))

        loss = criterion(logits_flat, target)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_GRAD)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader)


@torch.no_grad()
def evaluate_epoch(model, loader, criterion, vocab, device, use_bertscore=False):
    model.eval()
    total_loss = 0.0
    all_preds, all_refs = [], []

    pbar = tqdm(loader, desc="  Val  ", leave=False)
    for batch in pbar:
        image   = batch["image"].to(device)
        q_ids   = batch["q_ids"].to(device)
        q_attn  = batch["q_attn"].to(device)
        ans_ids = batch["ans_ids"].to(device)
        refs    = batch["raw_answer"]  # list of strings

        # Loss (teacher-forcing = 1.0 khi eval loss)
        logits = model(image, q_ids, q_attn, target_ids=ans_ids,
                       teacher_forcing_ratio=1.0)
        target = ans_ids[:, 1:].reshape(-1)
        loss   = criterion(logits.reshape(-1, logits.size(-1)), target)
        total_loss += loss.item()

        # Greedy decode cho metrics
        pred_ids = model(image, q_ids, q_attn, target_ids=None)  # (B, gen_len)
        preds = decode_predictions(pred_ids.cpu(), vocab)

        all_preds.extend(preds)
        all_refs.extend(refs)

    avg_loss = total_loss / len(loader)
    metrics  = compute_all_metrics(all_preds, all_refs, use_bertscore=use_bertscore)
    return avg_loss, metrics


def run_training(model, cfg, ckpt_dir, device, config_name):
    """
    Vòng lặp training đầy đủ với resume.
    config_name: "A1" hoặc "A2"
    """
    model = model.to(device)

    # Optimizer: tách LR cho PhoBERT và phần còn lại
    phobert_params = list(model.text_enc.bert.parameters())
    phobert_ids    = set(id(p) for p in phobert_params)
    other_params   = [p for p in model.parameters()
                      if id(p) not in phobert_ids and p.requires_grad]

    optimizer = AdamW([
        {"params": phobert_params, "lr": cfg.PHOBERT_LR},
        {"params": other_params,   "lr": cfg.LR},
    ], weight_decay=cfg.WEIGHT_DECAY)

    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx)

    # Resume nếu có checkpoint cũ
    start_epoch, best_loss = CheckpointManager.load_latest(
        model, optimizer, scheduler, ckpt_dir, device
    )

    print(f"\n{'='*55}")
    print(f"  Bắt đầu training {config_name}")
    print(f"  Epochs: {start_epoch} -> {cfg.EPOCHS} | Device: {device}")
    print(f"{'='*55}")

    history = []

    for epoch in range(start_epoch, cfg.EPOCHS):
        # Giảm dần teacher-forcing theo epoch
        tf_ratio = max(0.3, cfg.TEACHER_FORCING - epoch * 0.02)

        t0 = time.time()
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, cfg, device, tf_ratio
        )
        val_loss, metrics = evaluate_epoch(
            model, val_loader, criterion, vocab, device, use_bertscore=False
        )
        scheduler.step()
        elapsed = time.time() - t0

        is_best = val_loss < best_loss
        if is_best:
            best_loss = val_loss

        log = {
            "epoch":        epoch,
            "train_loss":   round(train_loss, 5),
            "val_loss":     round(val_loss, 5),
            "vqa_accuracy": round(metrics["vqa_accuracy"] * 100, 2),
            "bleu":         round(metrics["bleu"] * 100, 2),
            "rougeL":       round(metrics["rougeL"] * 100, 2),
            "meteor":       round(metrics["meteor"] * 100, 2),
            "elapsed_s":    round(elapsed, 1),
            "tf_ratio":     round(tf_ratio, 3),
        }
        history.append(log)

        print(
            f"Epoch {epoch:03d}/{cfg.EPOCHS-1} | "
            f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
            f"VQA: {metrics['vqa_accuracy']*100:.1f}% | "
            f"BLEU: {metrics['bleu']*100:.1f} | "
            f"ROUGE-L: {metrics['rougeL']*100:.1f} | "
            f"METEOR: {metrics['meteor']*100:.1f} | "
            f"({elapsed:.0f}s)"
        )

        # Chi luu khi co cai thien (tiết kiem dia), khong luu moi epoch
        if is_best:
            CheckpointManager.save(
                model, optimizer, scheduler, epoch, val_loss, best_loss,
                ckpt_dir, log, is_best=True
            )
        else:
            # Luu log va checkpoint phuc hoi sau moi SAVE_EVERY epoch
            if epoch % cfg.SAVE_EVERY == 0:
                CheckpointManager.save(
                    model, optimizer, scheduler, epoch, val_loss, best_loss,
                    ckpt_dir, log, is_best=False
                )
            else:
                # Chi ghi log, khong luu .pth
                log_path = os.path.join(ckpt_dir, "training_log.json")
                with open(log_path, "a", encoding="utf-8") as flog:
                    flog.write(json.dumps(log, ensure_ascii=False) + "\n")


    print(f"\nTraining {config_name} hoàn tất! Best Val Loss: {best_loss:.4f}")
    return history

print("Training loop sẵn sàng!")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Training loop sẵn sàng!


## 11. Train A1 (LSTM Decoder)

In [ ]:
print("Khởi tạo mô hình A1 (ResNet50 + PhoBERT + LSTM)...")
model_a1 = VQA_A1(cfg, vocab_size=len(vocab))

total_params = sum(p.numel() for p in model_a1.parameters())
train_params = sum(p.numel() for p in model_a1.parameters() if p.requires_grad)
print(f"Tổng params: {total_params:,} | Trainable: {train_params:,}")

history_a1 = run_training(
    model    = model_a1,
    cfg      = cfg,
    ckpt_dir = f"{CKPT_ROOT}/A1",
    device   = DEVICE,
    config_name = "A1",
)

Khởi tạo mô hình A1 (ResNet50 + PhoBERT + LSTM)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Tổng params: 172,935,818 | Trainable: 153,890,378
  [RESUME] Tiep tuc tu epoch 1 | Best Loss: 1.6566

  Bắt đầu training A1
  Epochs: 1 -> 5 | Device: cuda


  Train:   0%|          | 0/416 [00:00<?, ?it/s]

  Val  :   0%|          | 0/117 [00:00<?, ?it/s]

Epoch 001/4 | Train: 2.0023 | Val: 1.6186 | VQA: 40.6% | BLEU: 0.0 | ROUGE-L: 45.3 | METEOR: 21.5 | (856s)
  [LUU] ckpt_epoch_001.pth | Dia con: 58.2 GB
  [TOT NHAT] Mo hinh tot nhat! Val Loss: 1.6186


  Train:   0%|          | 0/416 [00:00<?, ?it/s]

  Val  :   0%|          | 0/117 [00:00<?, ?it/s]

## 12. Train A2 (Transformer Decoder)

In [ ]:
print("Khởi tạo mô hình A2 (ResNet50 + PhoBERT + Transformer)...")
model_a2 = VQA_A2(cfg, vocab_size=len(vocab))

total_params = sum(p.numel() for p in model_a2.parameters())
train_params = sum(p.numel() for p in model_a2.parameters() if p.requires_grad)
print(f"Tổng params: {total_params:,} | Trainable: {train_params:,}")

history_a2 = run_training(
    model    = model_a2,
    cfg      = cfg,
    ckpt_dir = f"{CKPT_ROOT}/A2",
    device   = DEVICE,
    config_name = "A2",
)

## 13. Đánh giá trên tập Test

In [ ]:
# --- 13. Đánh giá trên tập Test bằng Universal Evaluator ---

@torch.no_grad()
def evaluate_test_with_evaluator(model, test_loader, vocab, device, config_name, evaluator):
    """Đánh giá toàn diện trên tập test sử dụng UniversalEvaluator."""
    model.eval()
    all_preds, all_refs = [], []

    for batch in tqdm(test_loader, desc=f"Testing {config_name}"):
        image  = batch["image"].to(device)
        q_ids  = batch["q_ids"].to(device)
        q_attn = batch["q_attn"].to(device)
        refs   = batch["raw_answer"]

        # Inference: sinh câu trả lời
        pred_ids = model(image, q_ids, q_attn, target_ids=None)
        preds    = decode_predictions(pred_ids.cpu(), vocab)

        all_preds.extend(preds)
        all_refs.extend(refs)

    # Chạy Universal Evaluator
    eval_results = evaluator.evaluate_and_save(all_preds, all_refs, config_name)
    metrics = eval_results["metrics"]

    # (Tuỳ chọn) Lưu chi tiết predictions để so sánh thủ công (pred vs ref)
    out_detailed = {
        "config": config_name,
        "predictions": [{"pred": p, "ref": r} for p, r in zip(all_preds, all_refs)]
    }

    # --- TẠO THƯ MỤC LƯU THEO TÊN MÔ HÌNH (A1 / A2) CHO BẢN DETAILED ---
    model_metrics_dir = os.path.join(evaluator.base_metrics_dir, config_name)
    os.makedirs(model_metrics_dir, exist_ok=True)

    detail_path = os.path.join(model_metrics_dir, f"{config_name}_detailed_predictions.json")
    with open(detail_path, "w", encoding="utf-8") as f:
        json.dump(out_detailed, f, ensure_ascii=False, indent=2)

    return metrics, all_preds, all_refs


# --- Load best model và thực thi ---
def load_best_model(model_cls, cfg, vocab_size, ckpt_dir, device):
    model = model_cls(cfg, vocab_size).to(device)
    best_path = os.path.join(ckpt_dir, "best_model.pth")
    if os.path.exists(best_path):
        state = torch.load(best_path, map_location=device, weights_only=False)
        model.load_state_dict(state["model_state_dict"])
        print(f"Loaded best model từ {best_path} (epoch {state['epoch']})")
    else:
        print(f"Không tìm thấy best_model.pth tại {ckpt_dir}")
    return model

# 1. Khởi tạo Evaluator chung (truyền tham số base_metrics_dir mới)
evaluator = UniversalEvaluator(base_metrics_dir=f"{RESULT_ROOT}/metrics")

# 2. Đánh giá A1
print("\n--- ĐÁNH GIÁ MÔ HÌNH A1 (LSTM) ---")
best_a1 = load_best_model(VQA_A1, cfg, len(vocab), f"{CKPT_ROOT}/A1", DEVICE)
metrics_a1, preds_a1, refs_a1 = evaluate_test_with_evaluator(
    best_a1, test_loader, vocab, DEVICE, "A1", evaluator
)

# 3. Đánh giá A2
print("\n--- ĐÁNH GIÁ MÔ HÌNH A2 (Transformer) ---")
best_a2 = load_best_model(VQA_A2, cfg, len(vocab), f"{CKPT_ROOT}/A2", DEVICE)
metrics_a2, preds_a2, refs_a2 = evaluate_test_with_evaluator(
    best_a2, test_loader, vocab, DEVICE, "A2", evaluator
)

## 14. So sánh A1 vs A2 + Vẽ đồ thị

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'


def plot_training_curves(history_a1, history_a2, save_dir):
    """Vẽ learning curves so sánh A1 vs A2."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("So sánh A1 (LSTM) vs A2 (Transformer)", fontsize=16, fontweight="bold")

    metrics_to_plot = [
        ("train_loss",   "Train Loss",    False),
        ("val_loss",     "Val Loss",      False),
        ("vqa_accuracy", "VQA Accuracy (%)", True),
        ("bleu",         "BLEU (%)",      True),
        ("rougeL",       "ROUGE-L (%)",   True),
        ("meteor",       "METEOR (%)",    True),
    ]

    for ax, (key, label, is_pct) in zip(axes.flat, metrics_to_plot):
        if history_a1:
            epochs_a1 = [h["epoch"] for h in history_a1]
            vals_a1   = [h[key] for h in history_a1]
            ax.plot(epochs_a1, vals_a1, "b-o", markersize=4, label="A1 (LSTM)")
        if history_a2:
            epochs_a2 = [h["epoch"] for h in history_a2]
            vals_a2   = [h[key] for h in history_a2]
            ax.plot(epochs_a2, vals_a2, "r-s", markersize=4, label="A2 (Transformer)")
        ax.set_title(label)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(save_dir, "A1_vs_A2_training_curves.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu biểu đồ: {save_path}")


def plot_test_comparison(metrics_a1, metrics_a2, save_dir):
    """Bar chart so sánh test metrics A1 vs A2."""
    keys   = ["vqa_accuracy", "bleu", "rougeL", "meteor"]
    labels = ["VQA Acc", "BLEU", "ROUGE-L", "METEOR"]
    v1 = [metrics_a1.get(k, 0) * 100 for k in keys]
    v2 = [metrics_a2.get(k, 0) * 100 for k in keys]

    x  = np.arange(len(labels))
    w  = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    bars1 = ax.bar(x - w/2, v1, w, label="A1 (LSTM)",        color="steelblue",  alpha=0.85)
    bars2 = ax.bar(x + w/2, v2, w, label="A2 (Transformer)", color="tomato",     alpha=0.85)

    ax.set_title("So sánh A1 vs A2 trên tập Test", fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_ylabel("Score (%)")
    ax.legend(fontsize=11)
    ax.grid(axis="y", alpha=0.3)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    save_path = os.path.join(save_dir, "A1_vs_A2_test_comparison.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu biểu đồ: {save_path}")


# Vẽ
plot_training_curves(history_a1, history_a2, f"{RESULT_ROOT}/plots")
plot_test_comparison(metrics_a1, metrics_a2, f"{RESULT_ROOT}/plots")

## 15. Bảng tổng hợp kết quả

In [ ]:
import pandas as pd

def build_summary_table(metrics_a1, metrics_a2):
    rows = []
    for cfg_name, m in [("A1 (LSTM Decoder)", metrics_a1),
                        ("A2 (Transformer Decoder)", metrics_a2)]:
        row = {
            "Cấu hình":       cfg_name,
            "VQA Acc (%)": round(m.get("vqa_accuracy", 0) * 100, 2),
            "BLEU":           round(m.get("bleu", 0) * 100, 2),
            "ROUGE-L":        round(m.get("rougeL", 0) * 100, 2),
            "METEOR":         round(m.get("meteor", 0) * 100, 2),
            "BERTScore F1":   round(m.get("bertscore_f1", 0) * 100, 2),
        }
        rows.append(row)

    df = pd.DataFrame(rows).set_index("Cấu hình")
    print("\nBẢNG TỔNG HỢP KẾT QUẢ HƯỚNG A")
    print("=" * 70)
    print(df.to_string())
    print("=" * 70)

    # Lưu
    df.to_csv(os.path.join(RESULT_ROOT, "metrics", "summary_A1_A2.csv"))
    print("Đã lưu summary_A1_A2.csv")
    return df

summary_df = build_summary_table(metrics_a1, metrics_a2)

## 16. Demo dự đoán mẫu

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage

@torch.no_grad()
def demo_predictions(model_a1, model_a2, test_loader, vocab, device, n_samples=8):
    """Hiển thị n_samples ví dụ dự đoán so sánh A1 vs A2."""
    model_a1.eval()
    model_a2.eval()

    samples = []
    for batch in test_loader:
        image   = batch["image"].to(device)
        q_ids   = batch["q_ids"].to(device)
        q_attn  = batch["q_attn"].to(device)

        pred_a1 = decode_predictions(model_a1(image, q_ids, q_attn).cpu(), vocab)
        pred_a2 = decode_predictions(model_a2(image, q_ids, q_attn).cpu(), vocab)

        for i in range(len(batch["raw_answer"])):
            samples.append({
                "image_tensor": batch["image"][i],
                "question":     batch["raw_question"][i],
                "answer":       batch["raw_answer"][i],
                "pred_a1":      pred_a1[i],
                "pred_a2":      pred_a2[i],
            })
        if len(samples) >= n_samples:
            break

    samples = samples[:n_samples]
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle("Demo dự đoán: A1 (LSTM) vs A2 (Transformer)", fontsize=14, fontweight="bold")

    # Denormalize để hiện ảnh
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)

    for ax, s in zip(axes.flat, samples):
        img = s["image_tensor"] * std + mean
        img = img.permute(1, 2, 0).clamp(0, 1).numpy()
        ax.imshow(img)
        ax.axis("off")
        ok_a1 = "[DUNG]" if s["pred_a1"].lower() == s["answer"].lower() else "[SAI]"
        ok_a2 = "[DUNG]" if s["pred_a2"].lower() == s["answer"].lower() else "[SAI]"
        ax.set_title(
            f"Q: {s['question'][:40]}\n"
            f"Ref: {s['answer']}\n"
            f"A1: {s['pred_a1']} {ok_a1}\n"
            f"A2: {s['pred_a2']} {ok_a2}",
            fontsize=8, loc="left"
        )

    plt.tight_layout()
    save_path = os.path.join(RESULT_ROOT, "plots", "demo_predictions.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Đã lưu: {save_path}")


demo_predictions(best_a1, best_a2, test_loader, vocab, DEVICE, n_samples=8)

In [ ]:
print('Checkpoints lưu tại: ' + CKPT_ROOT)
print('Kết quả lưu tại:     ' + RESULT_ROOT)

---
## Tóm tắt Hướng A

| Module | A1 | A2 |
|---|---|---|
| **Image Encoder** | ResNet50 (pretrained, fine-tune layer4) | ResNet50 (pretrained, fine-tune layer4) |
| **Text Encoder** | PhoBERT (fine-tune) | PhoBERT (fine-tune) |
| **Fusion** | Concat + FC (2-layer MLP) | Concat + FC (2-layer MLP) |
| **Decoder** | LSTM (2 layers, hidden=512) | Transformer (3 layers, nhead=8) |

**Kết luận**: So sánh A1 vs A2 làm rõ ảnh hưởng của decoder architecture lên chất lượng sinh câu trả lời trong bài toán VQA tiếng Việt.